# Local Multi-Speaker Voice Processing Demo

This notebook runs the fully local voice processing pipeline from `audio.wav`, a local video file, or a YouTube URL.

- loads and normalizes audio to 16 kHz mono
- segments speech with Silero VAD
- extracts SpeechBrain ECAPA speaker embeddings
- clusters speakers with explicit agglomerative clustering
- transcribes with local Whisper
- emits speaker-labeled transcript JSON

Optional: if you provide a YouTube URL, the notebook downloads audio locally with `yt-dlp`, converts it to 16 kHz mono WAV with `ffmpeg`, and runs the same pipeline on that extracted audio. If you provide a local video file, it extracts audio with `ffmpeg`. If you also provide `user_voice.wav`, the pipeline will label matching turns as `USER`. Without it, output labels stay as `Speaker_0`, `Speaker_1`, etc.

Install the optional local speech stack before running model-backed cells:

```bash
pip install -e '.[voice-processing]'
```

In [1]:
from pathlib import Path

import json
import shutil
import subprocess
import sys


PROJECT_ROOT = Path.cwd()
DEFAULT_AUDIO_PATH = PROJECT_ROOT / "youtube_audio.wav"
if not DEFAULT_AUDIO_PATH.exists() and (PROJECT_ROOT / "tests" / "youtube_audio.wav").exists():
    DEFAULT_AUDIO_PATH = PROJECT_ROOT / "tests" / "youtube_audio.wav"
AUDIO_PATH = DEFAULT_AUDIO_PATH
YOUTUBE_URL = None  # Already downloaded from https://www.youtube.com/watch?v=gNA2fQ2KAIs
VIDEO_PATH = None  # Example: PROJECT_ROOT / "meeting.mp4"
YOUTUBE_COOKIES_FROM_BROWSER = None  # Example: "chrome", "safari", or "firefox" for restricted videos.
EXTRACTED_AUDIO_PATH = PROJECT_ROOT / "extracted_audio.wav"
YOUTUBE_DOWNLOAD_STEM = PROJECT_ROOT / "youtube_downloaded_audio"
YOUTUBE_AUDIO_PATH = PROJECT_ROOT / "youtube_audio.wav"
OPTIONAL_USER_VOICE_PATH = PROJECT_ROOT / "user_voice.wav"
USER_VOICE_PATH = OPTIONAL_USER_VOICE_PATH if OPTIONAL_USER_VOICE_PATH.exists() else None
OUTPUT_JSON_PATH = PROJECT_ROOT / "voice_transcript.json"

print("audio:", AUDIO_PATH)
print("youtube url:", YOUTUBE_URL)
print("video:", VIDEO_PATH)
print("youtube cookies browser:", YOUTUBE_COOKIES_FROM_BROWSER)
print("extracted audio:", EXTRACTED_AUDIO_PATH)
print("youtube audio:", YOUTUBE_AUDIO_PATH)
print("optional user voice:", USER_VOICE_PATH)
print("output:", OUTPUT_JSON_PATH)

audio: /Users/saketm10/Projects/audio_ml/test/youtube_audio.wav
youtube url: None
video: None
youtube cookies browser: None
extracted audio: /Users/saketm10/Projects/audio_ml/test/extracted_audio.wav
youtube audio: /Users/saketm10/Projects/audio_ml/test/youtube_audio.wav
optional user voice: None
output: /Users/saketm10/Projects/audio_ml/test/voice_transcript.json


In [2]:
def require_ffmpeg() -> None:
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg is required for media extraction. Install ffmpeg and rerun this cell.")


def convert_to_pipeline_wav(input_path: Path, audio_path: Path) -> Path:
    require_ffmpeg()
    command = [
        "ffmpeg",
        "-y",
        "-i",
        str(input_path),
        "-vn",
        "-ac",
        "1",
        "-ar",
        "16000",
        "-sample_fmt",
        "s16",
        str(audio_path),
    ]
    subprocess.run(command, check=True)
    return audio_path


def extract_audio_from_video(video_path: Path, audio_path: Path) -> Path:
    if not video_path.exists():
        raise FileNotFoundError(f"Missing video file: {video_path}")
    return convert_to_pipeline_wav(video_path, audio_path)


def extract_audio_from_youtube(youtube_url: str, audio_path: Path) -> Path:
    require_ffmpeg()
    if not youtube_url.startswith(("http://", "https://")) or "youtube" not in youtube_url and "youtu.be" not in youtube_url:
        raise ValueError(f"YOUTUBE_URL does not look like a full YouTube URL: {youtube_url!r}")

    if shutil.which("yt-dlp") is None:
        try:
            import yt_dlp  # noqa: F401
        except ImportError as error:
            raise RuntimeError("yt-dlp is required for YouTube input. Install with: pip install yt-dlp") from error

    for stale_path in YOUTUBE_DOWNLOAD_STEM.parent.glob(YOUTUBE_DOWNLOAD_STEM.name + ".*"):
        stale_path.unlink()

    command = [
        sys.executable,
        "-m",
        "yt_dlp",
        "--no-playlist",
        "--newline",
        "-f",
        "bestaudio/best",
        "--extract-audio",
        "--audio-format",
        "m4a",
        "-o",
        str(YOUTUBE_DOWNLOAD_STEM) + ".%(ext)s",
    ]
    if YOUTUBE_COOKIES_FROM_BROWSER:
        command.extend(["--cookies-from-browser", YOUTUBE_COOKIES_FROM_BROWSER])
    command.append(youtube_url)

    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        stdout_tail = result.stdout[-4000:]
        stderr_tail = result.stderr[-4000:]
        print("yt-dlp stdout:\n", stdout_tail)
        print("yt-dlp stderr:\n", stderr_tail)
        raise RuntimeError(
            "yt-dlp failed with exit code "
            f"{result.returncode}.\n\nSTDERR tail:\n{stderr_tail}\n\n"
            "Common fixes: update yt-dlp, verify the URL, or set "
            "YOUTUBE_COOKIES_FROM_BROWSER='chrome'/'safari'/'firefox' for restricted videos."
        )

    candidates = sorted(YOUTUBE_DOWNLOAD_STEM.parent.glob(YOUTUBE_DOWNLOAD_STEM.name + ".*"))
    if not candidates:
        raise FileNotFoundError("yt-dlp finished but no downloaded audio file was found.")
    downloaded_path = candidates[0]
    print("downloaded YouTube audio:", downloaded_path)
    return convert_to_pipeline_wav(downloaded_path, audio_path)

if YOUTUBE_URL:
    EFFECTIVE_AUDIO_PATH = extract_audio_from_youtube(YOUTUBE_URL, YOUTUBE_AUDIO_PATH)
elif VIDEO_PATH is not None:
    EFFECTIVE_AUDIO_PATH = extract_audio_from_video(Path(VIDEO_PATH), EXTRACTED_AUDIO_PATH)
elif AUDIO_PATH.exists():
    EFFECTIVE_AUDIO_PATH = AUDIO_PATH
else:
    raise FileNotFoundError(f"Missing input audio file: {AUDIO_PATH}. Set YOUTUBE_URL or VIDEO_PATH to extract audio.")

print("pipeline audio:", EFFECTIVE_AUDIO_PATH)

if USER_VOICE_PATH is None:
    print("Running clustering-only diarization. No USER enrollment file found.")
else:
    print("Running diarization with USER enrollment:", USER_VOICE_PATH)

pipeline audio: /Users/saketm10/Projects/audio_ml/test/youtube_audio.wav
Running clustering-only diarization. No USER enrollment file found.


In [3]:
import importlib.util

required_modules = {
    "torch": "torch",
    "torchaudio": "torchaudio",
    "silero_vad": "silero-vad",
    "speechbrain": "speechbrain",
    "whisper": "openai-whisper",
}

missing = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError(
        "Missing local voice-processing dependencies: "
        + ", ".join(missing)
        + ". Install with: pip install -e '.[voice-processing]'"
    )

print("All local model dependencies are importable.")

All local model dependencies are importable.


In [4]:
from input_audio_processing import PipelineConfig, VoiceProcessingPipeline

config = PipelineConfig(
    whisper_model_name="small",
    language=None,
    output_json_path=OUTPUT_JSON_PATH,
    vad_threshold=0.5,
    min_segment_s=0.6,
    max_segment_s=3.0,
    preferred_segment_s=2.0,
    user_similarity_threshold=0.7,
    speaker_assignment_strategy="unique_speaker_similarity",
    speaker_similarity_threshold=0.5,
    cluster_distance_threshold=0.4,
    asr_chunk_s=28.0,
)

print(config)
print("device:", config.resolve_device())

PipelineConfig(sample_rate=16000, vad_threshold=0.5, vad_min_silence_ms=250, vad_min_speech_ms=200, vad_max_speech_s=18.0, preferred_segment_s=2.0, min_segment_s=0.6, max_segment_s=3.0, embedding_batch_size=16, user_similarity_threshold=0.7, user_margin_threshold=0.04, speaker_assignment_strategy='unique_speaker_similarity', speaker_similarity_threshold=0.5, cluster_distance_threshold=0.4, cluster_min_size=1, whisper_model_name='small', whisper_device=None, whisper_compute_type='float16', asr_chunk_s=28.0, asr_chunk_padding_s=0.2, language=None, output_json_path=PosixPath('/Users/saketm10/Projects/audio_ml/test/voice_transcript.json'))
device: cpu


In [ ]:
pipeline = VoiceProcessingPipeline(config=config)
transcript = pipeline.run(EFFECTIVE_AUDIO_PATH, USER_VOICE_PATH)

print("Speaker assignment after similarity matching:")
for decision in pipeline.speaker_assignment_decisions:
    if decision.similarity is None or decision.confidence is None:
        print(f"{decision.provisional_speaker} assigned to {decision.assigned_speaker} (first unique speaker)")
        continue
    action = "matched existing" if decision.matched_existing else "created new speaker"
    print(
        f"{decision.provisional_speaker} assigned to {decision.assigned_speaker} "
        f"({action}, similarity={decision.similarity:.4f}, "
        f"threshold={decision.threshold:.4f}, confidence={decision.confidence:.4f})"
    )

transcript[:3]


In [ ]:
print(json.dumps(transcript, indent=2))

In [ ]:
for item in transcript:
    print(f"[{item['start']:.2f}-{item['end']:.2f}] {item['speaker']}: {item['text']}")

## Threshold Tuning

Default speaker assignment uses `unique_speaker_similarity`: each new segment is compared to the current unique speaker centroids in timeline order. If the best similarity is above `speaker_similarity_threshold`, it joins that speaker; otherwise a new `Speaker_N` is created.

If unknown speakers are merged together, raise `speaker_similarity_threshold`. If one speaker is split into many labels, lower it.

You can also switch to global agglomerative grouping with `speaker_assignment_strategy="agglomerative"`. In that mode, lower `cluster_distance_threshold` to split more speakers and raise it to merge more speakers.

If you add `user_voice.wav` and `USER` is over-assigned, raise `user_similarity_threshold`. If `USER` is missed, lower it in small steps such as `0.05`.

In [ ]:
if OUTPUT_JSON_PATH.exists():
    loaded = json.loads(OUTPUT_JSON_PATH.read_text(encoding="utf-8"))
    print(f"Loaded {len(loaded)} transcript turns from {OUTPUT_JSON_PATH}")
    print(loaded[:1])
else:
    print("Output JSON has not been written yet.")